# Comparison of SVM and Deep Learning Models
Performance metrics on the test set and spatio-temporal performance analysis

In [4]:
import warnings
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.svm import SVR, LinearSVR, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error,
)
import geopandas as gpd
import contextily as ctx
import shapely

import joblib

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODEL_DIR = Path("../models")
VISUALS_DIR = Path("../plots")
TARGET = "Total_Trip_Start" # Target variable for prediction

In [ ]:
# Load all models from the models directory
model_files = list(MODEL_DIR.glob("*.joblib"))

# Load the test set
df_test = pd.read_parquet("../data/prediction_split/res_8/df_test.parquet")


In [ ]:
# Feature columns for the different models
BASIC_FEATURES = [
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday", "day_of_week",
    "lat", "lon", "scikit_distance_to_loop", "base_demand"
]
POI_COLS = [c for c in df_test.columns if c.startswith("poi_cat_")]
WEATHER = ["2m_temp_c", "total_precip_mm", "snow_cov", "snow_depth", "wind_speed"]

feature_sets = {
    "basic": BASIC_FEATURES,
    "basic+poi": BASIC_FEATURES + POI_COLS,
    "basic+weather": BASIC_FEATURES + WEATHER,
    "basic+poi+weather": BASIC_FEATURES + POI_COLS + WEATHER,
}
# TODO: use this feature_sets dictionary to load the correct features for each model based on its name or metadata.
# For that lets use the names in our model names and match them to the keys in this dictionary. 
# For example, if a model name contains "basic+poi", we will use the corresponding feature set from this dictionary.

In [7]:
# Inspect the loaded models
for model_file in model_files:
    model = joblib.load(model_file)
    print(f"Loaded model: {model_file.name}")
    print(f"Model type: {type(model)}")
    if hasattr(model, 'best_params_'):
        print(f"Best parameters: {model.best_params_}")
    if hasattr(model, 'best_score_'):
        print(f"Best score: {model.best_score_}")
    if hasattr(model, 'X_test') and hasattr(model, 'y_test'):
        print(f"Test set shape: {model.X_test.shape}, {model.y_test.shape}")
    print("-" * 40)


Loaded model: svr_best_8.joblib
Model type: <class 'dict'>
----------------------------------------
Loaded model: svr_best.joblib
Model type: <class 'dict'>
----------------------------------------
Loaded model: svr_best_7.joblib
Model type: <class 'dict'>
----------------------------------------


In [ ]:
# Create a DataFrame to store model performance metrics
performance_metrics = []

# Let the test set run through all models and store the performance metrics
for model_file in model_files:
    model_bundle = joblib.load(model_file)
    model = model_bundle["model"]
    # Fit the features based on the model name
    if "basic+poi+weather" in model_file.name:
        feature_set = feature_sets["basic+poi+weather"]
    elif "basic+poi" in model_file.name:
        feature_set = feature_sets["basic+poi"]
    elif "basic+weather" in model_file.name:
        feature_set = feature_sets["basic+weather"]
    elif "basic" in model_file.name:
        feature_set = feature_sets["basic"]
    else:
        raise ValueError(f"Unknown feature set for model: {model_file.name}")
    
    # Features for the test set
    X_test = df_test[feature_set].values
    # Target column for the test set
    y_test = df_test[TARGET].values

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate performance metrics
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    # Store the metrics in the DataFrame
    performance_metrics.append({
        "model_file": model_file.name,
        "mae": mae,
        "r2": r2,
        "mape": mape,
    })

# Convert the performance metrics list to a DataFrame
performance_df = pd.DataFrame(performance_metrics)
performance_df = performance_df.sort_values(by="model_file").reset_index(drop=True)
performance_df

KeyError: 'X_test'

## Performance Metrics Table
Here we compare our performance metrics on the test set for multiple SVM and NN models

## Performance Visualization

In [ ]:
# fig-svm-pred-vs-actual - Predicted vs actual for the best SVR model
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best_svr, alpha=0.2, s=8, color="royalblue")
lim = y_test.max()
plt.plot([0, lim], [0, lim], "r--", lw=2)
plt.title(f"{best_row_svr}: Actual vs. Predicted Demand")
plt.xlabel("Actual Trip Count")
plt.ylabel("Predicted Trip Count")
plt.tight_layout()
plt.show()

In [ ]:
# fig-nn-pred-vs-actual - Predicted vs actual for the best NN model
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best_nn, alpha=0.2, s=8, color="royalblue")
lim = y_test.max()
plt.plot([0, lim], [0, lim], "r--", lw=2)
plt.title(f"{best_row_nn}: Actual vs. Predicted Demand")
plt.xlabel("Actual Trip Count")
plt.ylabel("Predicted Trip Count")
plt.tight_layout()
plt.show()

## Spatio Performance Analytics

In [ ]:
# Community area outlines, used as a spatial reference overlay in the choropleths.
community_areas = gpd.read_file(Path("..") / "data" / "chicago_community_areas.gpkg")

def get_hexagon_grid(data, h3_col="h3_index"):
    """
    Build a GeoDataFrame of H3 hexagon polygons, preserving all columns from the source.
    """
    def _to_polygon(cell):
        boundary = h3.cell_to_boundary(cell)
        return shapely.geometry.Polygon([(lng, lat) for lat, lng in boundary])

    if isinstance(data, pd.Series):
        h3_list = list(data)
        df = pd.DataFrame({"h3_index": h3_list})
    else:
        h3_list = list(data[h3_col])
        df = data.reset_index(drop=True)
    geometries = [_to_polygon(c) for c in h3_list]
    return gpd.GeoDataFrame(df, geometry=geometries, crs="EPSG:4326")


def create_choropleth_plot(gdf, value_col="poi_count", title="", boundaries=None,
                           cmap="YlOrRd", figsize=(6, 8), k=7):
    """
    Create a static matplotlib choropleth of hexagon values with Chicago basemap.
    """
    if boundaries is None:
        boundaries = community_areas
    fig, ax = plt.subplots(figsize=figsize)
    gdf.to_crs(epsg=3857).plot(
        ax=ax, column=value_col, cmap=cmap, scheme="quantiles", k=k,
        edgecolor="none", alpha=0.8, legend=True, zorder=1,
        legend_kwds=dict(title=value_col.replace("_", " ").title(),
                         loc="lower right", fontsize=8, title_fontsize=9),
        missing_kwds=dict(color="lightgrey", label="No data"),
    )
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, zoom=11)
    boundaries.to_crs(epsg=3857).boundary.plot(
        ax=ax, color="#222", linewidth=1.0, alpha=0.9, zorder=3)
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    return fig, ax

In [ ]:
#| label: tbl-svm-hex-performance
# Per-hexagon performance for the best model
df_eval = df_test[["h3_index", TARGET]].copy()
df_eval["y_pred"] = y_pred_best
df_eval["abs_err"] = (df_eval[TARGET] - df_eval["y_pred"]).abs()
df_eval["sq_err"] = (df_eval[TARGET] - df_eval["y_pred"]) ** 2

grp = df_eval.groupby("h3_index")
hex_perf = grp.agg(
    n_obs=(TARGET, "size"),
    mean_actual=(TARGET, "mean"),
    var_actual=(TARGET, "var"),
    mae=("abs_err", "mean"),
    mse=("sq_err", "mean"),
)
hex_perf["rmse"] = np.sqrt(hex_perf["mse"])
ss_res = grp["sq_err"].sum()
ss_tot = hex_perf["var_actual"].fillna(0) * (hex_perf["n_obs"] - 1)
hex_perf["r2"] = 1 - ss_res / ss_tot.replace(0, np.nan)
hex_perf["rel_error"] = hex_perf["mae"] / hex_perf["mean_actual"].replace(0, np.nan)
hex_perf = hex_perf.drop(columns=["var_actual", "mse"]).reset_index()
hex_perf_gdf = get_hexagon_grid(hex_perf, h3_col="h3_index")

busy = hex_perf[hex_perf["mean_actual"] > 1]
n_bad = (busy["r2"] < -0.3).sum()
#print(f"{len(hex_perf)} hexagons evaluated; "
#      f"{hex_perf['r2'].notna().sum()} have enough variation for an R2.")
#print(f"Busy cells (mean>1) with R2 < -0.3: {n_bad}  (original model had several)")
#print("\nWorst-predicted busy cells (lowest R2):")
#print(busy.nsmallest(5, "r2")[["h3_index", "n_obs", "mean_actual", "mae", "rmse", "r2"]]
#      .to_string(index=False))

### SVR - Support Vector Regression

In [ ]:
# fig-svm-hex-mae - MAE per hexagon for the best SVR model
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='mae',
    title='SVR absolute error per hexagon (MAE, trips/hour)\nred = larger prediction error',
    cmap='YlOrRd',
    k=7,
)
plt.show()

In [ ]:
# fig-svm-hex-mre - MRE per hexagon for the best SVR model
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='rel_error',
    title='SVR relative error per hexagon (MRE, trips/hour)\nred = larger prediction error',
    cmap='YlOrRd',
    k=7,
)
plt.show()

In [ ]:
# fig-svm-hex-r2 - R² per hexagon for the best SVR model
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='r2',
    title='SVR prediction quality per hexagon (R²)\ngreen = well predicted, grey = too little demand to assess',
    cmap='RdYlGn',
    k=7,
)
plt.show()

### Deep Learning - Neural Network